# Chapter 3 Solutions: Machine Learning and the Train-Test Split

Practise splitting, repeating, noise features and leakage by calculating on a new quiz dataset (`quiz03_data.csv`).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/sol-03-machine-learning.ipynb)

Lecture: [en-03-machine-learning](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-03-machine-learning.md)  
Labs: [LM301](https://rathachai.github.io/DA-LAB/learnings/linear/lm301.html) | [LM302](https://rathachai.github.io/DA-LAB/learnings/linear/lm302.html)

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

## How to use
This notebook contains **sample solutions**. Try the exercise notebook first, then compare. Run the SETUP cell first, then the cells in order. Your numbers may differ slightly if you choose other features or seeds.

## SETUP
The quiz data are in the CSV file `quiz03_data.csv`: 150 samples with the columns `id, x1, x2, x3, x4, x5, y`. The true rule used to create them is `y = 4 + 2.5*x1 - 1.8*x2 + noise`. The columns `x3, x4, x5` are pure noise and carry no information.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/quiz03_data.csv"
df = pd.read_csv(url)
print(df.shape)
df

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 1: One split, two errors**

Split `df` 70 / 30 with `random_state=7`. Fit a `LinearRegression` on `x1, x2` using the train part only. Report **train and test RMSE and MAE** in a small table.

*Hint: `sklearn.model_selection.train_test_split`, `LinearRegression`, `sklearn.metrics.root_mean_squared_error` and `mean_absolute_error`; build the table with `pd.DataFrame`.*

</div>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

X = df[["x1", "x2"]]
y = df["y"]
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=7)

model = LinearRegression().fit(X_train, y_train)
rows = []
for name, Xs, ys in [("train", X_train, y_train), ("test", X_test, y_test)]:
    p = model.predict(Xs)
    rows.append({"set": name, "RMSE": root_mean_squared_error(ys, p), "MAE": mean_absolute_error(ys, p)})
pd.DataFrame(rows).round(3)

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** Test error (RMSE 1.66) is a little higher than train error (1.45): the model is fitted on the train rows, so train looks slightly better. The true noise level is about 1.5, so this is a healthy gap.

</div>

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 2: Effect of the train ratio**

For train ratios 10, 20, ..., 90 %, repeat 30 random splits. Fit on `x1, x2` and record the test RMSE. Plot the **mean test RMSE with a plus/minus std band** against the ratio. At which ratio is the spread largest? Why?

*Hint: `ShuffleSplit(n_splits=30, train_size=r/100, random_state=0)` gives the 30 splits; collect rows in a `pd.DataFrame`, then `groupby("ratio").agg(...)`; plot with `plt.errorbar`.*

</div>

In [ ]:
from sklearn.model_selection import ShuffleSplit

rows = []
for ratio in range(10, 100, 10):
    ss = ShuffleSplit(n_splits=30, train_size=ratio / 100, random_state=0)
    for tr, te in ss.split(X):
        m = LinearRegression().fit(X.iloc[tr], y.iloc[tr])
        rows.append({"ratio": ratio, "rmse": root_mean_squared_error(y.iloc[te], m.predict(X.iloc[te]))})
res = pd.DataFrame(rows).groupby("ratio")["rmse"].agg(["mean", "std"])
print(res.round(3))

plt.errorbar(res.index, res["mean"], yerr=res["std"], marker="o", capsize=3)
plt.xlabel("train ratio (%)"); plt.ylabel("test RMSE (mean ± std)"); plt.grid(alpha=0.3)
plt.show()

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** The mean test RMSE falls from about 1.65 (10 % train) to about 1.51 (70-80 %), then the **spread (std) grows** from 0.07 to 0.26 at 90 %. With few train rows the model is poor; with few test rows the score is noisy. The spread is largest at 90 % because only 15 test rows are left.

</div>

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 3: Do noise features hurt?**

Compare the test RMSE of two models: (A) `x1, x2` and (B) `x1..x5`. Do it at **20 %** and at **80 %** train, averaging over 30 random splits each. Show a 2 by 2 table. When does the noise hurt more?

*Hint: Use `ShuffleSplit`, `LinearRegression`, `root_mean_squared_error`; summarise with `pd.DataFrame` / `groupby` (rows A and B, columns 20 % and 80 %).*

</div>

In [ ]:
rows = []
for ratio in (20, 80):
    ss = ShuffleSplit(n_splits=30, train_size=ratio / 100, random_state=0)
    for name, cols in [("A: x1,x2", ["x1", "x2"]), ("B: x1..x5", ["x1", "x2", "x3", "x4", "x5"])]:
        for tr, te in ss.split(df):
            m = LinearRegression().fit(df.iloc[tr][cols], y.iloc[tr])
            rows.append({"model": name, "train %": ratio,
                         "rmse": root_mean_squared_error(y.iloc[te], m.predict(df.iloc[te][cols]))})
table = pd.DataFrame(rows).groupby(["model", "train %"])["rmse"].mean().unstack().round(3)
table

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** At 20 % train: A = 1.578, B = 1.685, so the noise features **hurt** (+0.11). At 80 % train: A = 1.507, B = 1.517, almost no harm. Spare features let the model memorise noise, and a small train set is easiest to memorise.

</div>

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 4: Leakage by feature selection**

Simulate (seeded) 100 samples with 200 pure-noise features and a pure-noise target `y`. Compare, over 30 random 50 / 50 splits: **(A)** pick the 5 features most related to `y` using ALL rows, then split, fit `LinearRegression` and compute the test R2; **(B)** split first, pick the 5 features on the train rows only, then fit and compute the test R2. Show the mean test R2 of A and B in a small table, and explain in 2 sentences why A looks better and why B is honest. The same idea explains why pre-processing (scaling, imputing) must be fitted on the training set only.

*Hint: `SelectKBest(f_regression, k=5)` (or rank features with `df.corrwith(y).abs()`), `train_test_split` / `ShuffleSplit`, `LinearRegression`, `sklearn.metrics.r2_score`; `pd.DataFrame` for the table.*

</div>

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import r2_score

rng = np.random.default_rng(0)
Xn = rng.normal(size=(100, 200))
yn = rng.normal(size=100)

r2_a, r2_b = [], []
for seed in range(30):
    idx = np.random.default_rng(seed).permutation(100)
    tr, te = idx[:50], idx[50:]

    # A: select the 5 features using ALL rows (leak), then split
    cols = SelectKBest(f_regression, k=5).fit(Xn, yn).get_support()
    m = LinearRegression().fit(Xn[tr][:, cols], yn[tr])
    r2_a.append(r2_score(yn[te], m.predict(Xn[te][:, cols])))

    # B: select the 5 features using only the train rows (honest)
    cols = SelectKBest(f_regression, k=5).fit(Xn[tr], yn[tr]).get_support()
    m = LinearRegression().fit(Xn[tr][:, cols], yn[tr])
    r2_b.append(r2_score(yn[te], m.predict(Xn[te][:, cols])))

print("A: select first (leak)   mean test R2 =", round(np.mean(r2_a), 3))

print("B: split first (honest)  mean test R2 =", round(np.mean(r2_b), 3))

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** A (select first) gives test R2 about +0.15, B (split first) about -0.34. The data are pure noise, so the honest answer is R2 of 0 or below. In A the selection already saw the test rows, so it picked features that happen to fit them. Always split first, then select, scale or impute using the train rows only.

</div>

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 5: Pen-and-paper diagnosis**

A model has **train RMSE = 1.4** and **test RMSE = 4.9**. (a) Compute the ratio test / train. (b) Compute the gap. (c) Decide: underfitting, good fit or overfitting? Another model has train RMSE = 5.1 and test RMSE = 5.3: decide again. Use the cell below as a calculator and write the reasons as comments.

*Hint: Ratio = test / train; a calculator or `pd.DataFrame` is enough. Large gap means overfitting; both errors high and close means underfitting (if they are also high compared with the noise level).*

</div>

In [ ]:
ratio = 4.9 / 1.4
gap = 4.9 - 1.4
print("model 1: ratio =", round(ratio, 2), " gap =", round(gap, 2))
print("model 2: ratio =", round(5.3 / 5.1, 2), " gap =", round(5.3 - 5.1, 2))
# Model 1: test is 3.5x the train error -> large gap -> OVERFITTING (it memorised the train rows).
# Model 2: errors are high but almost equal -> no overfitting; if 5 is large compared with the noise level
#          it is UNDERFITTING, otherwise it is a good (honest) fit.

<div style="border:2px solid #16a34a; border-radius:8px; padding:12px; background:#f0fdf4">

**Solution notes.** Model 1: ratio 3.5 and gap 3.5, so it is **overfitting** (it memorised the train rows). Model 2: ratio 1.04 and gap 0.2, so train and test agree; no overfitting. Whether it is a good fit or underfitting depends on whether 5 is large compared with the noise level of the problem.

</div>

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Exercise 6: Your conclusion**

Write 3 to 5 sentences: What did the split ratio do to the score and its spread? What did the noise features do? Why must pre-processing be fitted on train only?

*Hint: Use your numbers from Exercises 1 to 5.*

</div>

<div style="border:2px solid #2563eb; border-radius:8px; padding:12px; background:#eff6ff">

**Your conclusion**

Increasing the train ratio lowered the mean test error until about 70 %, but the spread of the score grew when the test set became small (90 % train). The noise features `x3-x5` raised the test error mainly when the train set was small (20 %), because spare dials let the model memorise noise. Any step that learns from data (feature selection, scaling, imputing) must be fitted on the train rows only; otherwise the test score is optimistic (Exercise 4: +0.15 vs -0.34).

</div>

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io